# Experiment 3

**Steps 4, 5 and 6 of 8 — portfolio, backtest, attribution — for one idea.**

**It produces** a book in `Portfolio/`, a track record in `Backtest/` and a breakdown of the
return in `Attribution/`. **It prevents** a good signal in a portfolio nobody could hold, paper
returns that real trading would have erased, and factor beta sold as alpha.

The hypothesis is in [`BLUEPRINT_3.md`](BLUEPRINT_3.md), **written before this notebook's rule**.
The running log is [`JOURNAL_3.md`](JOURNAL_3.md); the results that survive are in
[`FINDINGS_3.md`](FINDINGS_3.md).

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_3.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

**What this notebook does not do.** It does not download anything, profile the universe, or compute
a signal. A number about the data itself belongs in the Universe or Data stage — that separation is
what keeps every experiment comparable, because all of them read the identical panel.

**Four modules beside this notebook are shared by every experiment** — `securities_panel.py`,
`portfolio_construction.py`, `backtest_engine.py` and `attribution_analysis.py`, one per Lab
library — and no strategy column is named in any of them. The benchmark loads the panel through
`securities_panel.py` like every later experiment, so the comparison is on the rule and nothing
else.

## The section contract

Every experiment notebook has the same shape, so anyone who has read one can read all of them.
**Everything below section 2 is strategy-agnostic**, given the three objects that section produces.

| Section | Contains |
| --- | --- |
| 0 · Setup | paths, and **the strategy's columns** — the only strategy names in this notebook outside the rule |
| 1 · The panel | load the refined files and reshape them |
| 2 · The rule | selection, sizing, timing. **The one cell you write** |
| 2.1 · Invariants | what every rule must pass, whatever it is |
| 3 · Construction | the book, and the diagnostics a person would run it on |
| 4 · Backtest | one engine pass, guarded import, reports-and-skips without a licence |
| 5 · Attribution | where the return came from, guarded the same way |
| 6 · Counterfactuals | the arms that price who earned the idiosyncratic share |
| 7 · Verdict | what it concluded, **in words** |
| Handoff | what the next stage consumes, and what this one left open |
| 8 · Verify | assertions that raise when the output is wrong: the invariants, the weight file read back, the days each engine run valued against its window's trading days |

## 0 · Setup

Paths, and the strategy's columns. **Declare them here, not in a shared module**, so a signal never
becomes every later experiment's default without anyone deciding it. Read only the columns the
strategy consumes.

Three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Basis | Why |
| --- | --- | --- |
| Daily mark | dividend-and-split adjusted close | total-return valuation between rebalances |
| Fill | dividend-and-split adjusted VWAP | the price a trade actually gets |
| Commission | **unadjusted** VWAP | per-share cents ride on the unadjusted share count |

A provider may return its VWAP columns as null, which is why the Curator reconstructs both VWAPs as
`c_*`, and nothing below reads a provider's own VWAP column, whichever provider `Data/curator.py`
asks for.

In [ ]:
# EXAMPLE-ONLY CELL
import json
import os
import pathlib
import sys

import pandas

NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = next(
    parent
    for parent in (NOTEBOOK_DIRECTORY, *NOTEBOOK_DIRECTORY.parents)
    if (parent / "Universe").is_dir()
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))
sys.path.insert(0, str(REPOSITORY_ROOT / "Data"))

import attribution_analysis  # noqa: E402 - the paths above have to exist first
import backtest_engine  # noqa: E402
import hand_supplied  # noqa: E402
import portfolio_construction  # noqa: E402
import securities_panel  # noqa: E402

EXPERIMENT_DIRECTORY = REPOSITORY_ROOT / "Experiments" / "Experiment_3"

# The strategy's columns, named here and nowhere else. Three prices do three different jobs.
SIGNAL_COLUMN = "r_momentum_12_1"
RANKING_COLUMN = "r_liquidity_rank"
MARK_COLUMN = "m_close_dividend_and_split_adjusted"
FILL_COLUMN = "c_vwap_dividend_and_split_adjusted"
TRADED_VALUE_COLUMN = "c_daily_traded_value"

# The rule's settings, from BLUEPRINT_3.md of 2026-09-24, committed before this cell was written.
BOOK_SIZE = 20
POOL_SIZE = 100
LOOKBACK_MONTHS = 12
SKIP_MONTHS = 1
TRADING_DAYS_PER_MONTH = 21
FREQUENCY = "month"
# The test window: the cash proxy's first price to the experiment's end; nothing later is read.
TEST_START = pandas.Timestamp("2002-07-30")
TEST_END = pandas.Timestamp("2026-06-01")
WINDOW_END = TEST_END
COVERAGE_REQUIRED = 0.95
# The kill switch's three sub-periods, each priced as a window of its own.
SUB_PERIODS = (
    (pandas.Timestamp("2002-07-30"), pandas.Timestamp("2008-12-31")),
    (pandas.Timestamp("2009-01-02"), pandas.Timestamp("2016-12-30")),
    (pandas.Timestamp("2017-01-03"), pandas.Timestamp("2026-06-01")),
)
# Prediction 3: the months after the 2009 low, on return, the rule against its control.
REBOUND_START = pandas.Timestamp("2009-03-02")
# The owner's first window, priced as description beside the test.
EARLY_END = pandas.Timestamp("2016-12-30")
REBOUND_END = pandas.Timestamp("2009-12-31")
SHARPE_MARGIN = 0.03
CAGR_MARGIN_POINTS = 0.5
# Criterion 3: the rule ahead of its control on Sharpe and on CAGR in at least this many cells.
CELLS_REQUIRED = 8
SWEEP = (
    ("lookback 6 months", {"lookback": 6}),
    ("lookback 9 months", {"lookback": 9}),
    ("12 months, no skip", {"skip": 0}),
    ("pool 50", {"pool_size": 50}),
    ("pool 150", {"pool_size": 150}),
    ("pool 200", {"pool_size": 200}),
    ("book size 10", {"book_size": 10}),
    ("book size 30", {"book_size": 30}),
    ("quarterly", {"frequency": "quarter"}),
    ("acted on 5 days late", {"delay": 5}),
)
# The exclusions are Experiment 1's two tests, read from the register the universe notebook wrote,
# and its identity check, read from the profile cache; none is chosen here.
BLOCKING_CHECKS_USED = (
    "missing file",
    "impossible daily move (the adjusted price multiplies by more than six)",
)
PROFILE_CACHE_PATH = pathlib.Path("Universe/Provider_Cache/profiles.json")
seed = pandas.read_csv("Universe/Investable_Universe.csv")
seed_providers = (
    seed["provider"].fillna("")
    if "provider" in seed.columns
    else pandas.Series("", index=seed.index)
)
SHARADAR_IDENTIFIERS = frozenset(seed.loc[seed_providers == "sharadar", "main_identifier"])

print(f"repository root: {REPOSITORY_ROOT}")
print(f"engine installed: {backtest_engine.ENGINE_INSTALLED}")
print(f"attribution installed: {attribution_analysis.LIBRARY_INSTALLED}")
print(f"seed: {len(seed)} identifiers, {len(SHARADAR_IDENTIFIERS)} of them Sharadar's")

## 1 · The panel

Load the refined files, resolve **one position per security**, and reshape to matrices.

A point-in-time universe contains renamed securities: two identifiers sharing one identity, each
carrying part of the history. Left alone they are two independent positions and the book
double-counts at the changeover. Key positions by a stable identity — an ISIN where the seed
carries one — falling back to the identifier itself, and where two legs overlap on a date let the
leg still reporting later win.

The long panel then becomes one wide `dates x securities` matrix per input, which is what makes the
whole rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
# EXAMPLE-ONLY CELL
matrices = securities_panel.load_matrices((
    SIGNAL_COLUMN,
    RANKING_COLUMN,
    MARK_COLUMN,
    FILL_COLUMN,
    TRADED_VALUE_COLUMN,
))
# Each column comes back on the dates it has values; every matrix is put on the price calendar, and
# nothing after the window's end is kept, so the held-out months cannot reach a single cell below.
calendar = matrices[MARK_COLUMN].index
in_panel = calendar <= WINDOW_END
mark = matrices[MARK_COLUMN].loc[in_panel]
signal = matrices[SIGNAL_COLUMN].reindex(index=mark.index, columns=mark.columns)
ranking = matrices[RANKING_COLUMN].reindex(index=mark.index, columns=mark.columns)
fill = matrices[FILL_COLUMN].reindex(index=mark.index, columns=mark.columns)
traded_value = matrices[TRADED_VALUE_COLUMN].reindex(index=mark.index, columns=mark.columns)
returns = mark.pct_change(fill_method=None)

# Membership, point in time: the index's own daily holdings, read as the desk ships them and mapped
# onto positions rather than listings, because a company that changed ticker is one position here.
position_keys = securities_panel.read_position_keys()
holdings = hand_supplied.read_benchmark_holdings()
holdings.columns = [position_keys.get(column, column) for column in holdings.columns]
index_weight = holdings.T.groupby(level=0).sum().T
membership = (index_weight > 0).reindex(
    index=mark.index,
    columns=mark.columns,
).ffill()
membership = membership.where(membership.notna(), False).astype(bool)

# The cash proxy's own return, for the books' cash.
cash_file = backtest_engine.MARKET_DATA_DIRECTORY / f"{backtest_engine.CASH_IDENTIFIER}.csv"
cash_prices = pandas.read_csv(
    cash_file,
    usecols=["m_date", MARK_COLUMN],
    parse_dates=["m_date"],
    index_col="m_date",
)[MARK_COLUMN]
cash_returns = cash_prices.reindex(mark.index).pct_change(fill_method=None).fillna(0.0)

# The exclusions, fixed by the blueprint's tests before the rule: listed in JOURNAL_2.md.
issues = pandas.read_csv("Universe/Data_Issues.csv").set_index("check")
failing = set()

for check in BLOCKING_CHECKS_USED:
    listed = issues.loc[check, "identifiers"]

    if isinstance(listed, str):
        failing.update(listed.split())

profiles = (
    json.loads(PROFILE_CACHE_PATH.read_text(encoding="utf-8"))
    if PROFILE_CACHE_PATH.is_file()
    else {}
)
two_companies = {
    identifier
    for identifier in seed["main_identifier"]
    if identifier not in SHARADAR_IDENTIFIERS
    and (profiles.get(identifier) or {}).get("symbol") not in (None, identifier)
}
EXCLUDED_IDENTIFIERS = tuple(sorted(failing | two_companies))
excluded = [identifier for identifier in EXCLUDED_IDENTIFIERS if identifier in mark.columns]

# Coverage, on every date before any book is built: the share of the index's weight in members with
# a price and a fill price that day, excluded names counted as unpriced.
priced = (mark.notna() & fill.notna()).drop(columns=excluded, errors="ignore")
coverage_dates = index_weight.index.intersection(mark.index)
weights_on_dates = index_weight.loc[coverage_dates]
priced_on_dates = priced.reindex(
    index=coverage_dates,
    columns=weights_on_dates.columns,
    fill_value=False,
)
coverage = weights_on_dates.where(priced_on_dates, 0.0).sum(axis=1) / weights_on_dates.sum(axis=1)
in_test = coverage[(coverage.index >= TEST_START) & (coverage.index <= TEST_END)]
below = in_test[in_test < COVERAGE_REQUIRED]
print(f"panel: {mark.shape[0]} dates x {mark.shape[1]} positions, to {mark.index.max().date()}")
print(f"index membership known from {holdings.index.min().date()} to {holdings.index.max().date()}")
print(f"excluded by name ({len(EXCLUDED_IDENTIFIERS)}): {', '.join(EXCLUDED_IDENTIFIERS)}")
print(f"  of them carrying two companies under one identifier: {sorted(two_companies)}")
print(f"coverage inside the test: lowest {in_test.min():.4f} on {in_test.idxmin().date()}; "
      f"dates below {COVERAGE_REQUIRED:.0%}: {len(below)}")

## 2 · The rule — the one cell you write

Three statements, in order: **who is eligible**, **how much of each**, and **when to trade**.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates x securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | a date index | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES x securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy. The residual becomes cash in section 3.1, parked in a real priced instrument,
because the engine's weight file has no cash row of its own.

**Sizing is a seam, not a decision buried in the rule.** Hand the eligible set and a returns
history that has already been cut off before today to a weighting function in
`portfolio_construction.py`, and swapping equal weight for inverse volatility, hierarchical risk
parity or any other method of the KaxaNuk Portfolio Construction library is one line — the module
builds the library's method on that cut history, one rebalance date at a time. That is what makes
two experiments comparable rather than merely adjacent.

**Trade only when something changed.** A signal that has not moved is not a reason to pay
commission.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t-1*, and the fill
happens at *t*'s price — a full day between the signal and the fill.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists.

In [ ]:
# EXAMPLE-ONLY CELL
# The rule, from BLUEPRINT_3.md: of the hundred most traded members, the twenty with the highest
# twelve-month return before the latest month, equally weighted, re-struck on the first trading day
# of each month. Written after the blueprint was committed, which the history shows.
tradable = mark.notna() & fill.notna()
tradable_members = (tradable & membership).drop(columns=excluded, errors="ignore")
candidates = tradable_members.reindex(columns=mark.columns, fill_value=False).astype(bool)


def momentum_score(lookback, skip):
    """
    The return over `lookback` months before the latest `skip` months, on the adjusted close.

    The rule's own setting reads the refinery's column; a perturbed one is computed here the same
    way, on the same prices, because the refinery carries one window.
    """
    if (lookback, skip) == (LOOKBACK_MONTHS, SKIP_MONTHS):
        return signal

    recent = mark.shift(skip * TRADING_DAYS_PER_MONTH)
    distant = mark.shift(lookback * TRADING_DAYS_PER_MONTH)

    return recent / distant - 1


ALPHABETICAL = sorted(mark.columns)


def ranked(frame):
    """Each date's values ranked highest first, a tie broken by main_identifier, alphabetically."""
    in_order = frame[ALPHABETICAL].rank(axis=1, ascending=False, method="first")

    return in_order.reindex(columns=mark.columns)


def build_book(
    start,
    end,
    book_size=BOOK_SIZE,
    pool_size=POOL_SIZE,
    lookback=LOOKBACK_MONTHS,
    skip=SKIP_MONTHS,
    frequency=FREQUENCY,
    delay=0,
    use_momentum=True,
    whole_pool=False,
    young_listings=False,
    rebalance_dates=None,
):
    """
    The book over a window, re-struck in full on each rebalance date and left to drift between.

    `use_momentum=False` is the control: the pool's most traded members with a momentum value, in
    the ranking's place, on the `rebalance_dates` it is given; `young_listings=True` lifts that
    restriction, the diagnostic. `whole_pool=True` is the null. The lag is taken before the window
    is cut, so the first day reads the close before it. `delay` fills each decision that many
    trading days after its rebalance date, on the same prior close.
    """
    in_window = (mark.index >= start) & (mark.index <= end)
    liquidity_order = ranked(ranking.where(candidates))
    pool = liquidity_order <= pool_size

    if whole_pool:
        chosen = pool
        size = pool_size
    elif use_momentum:
        chosen = ranked(momentum_score(lookback, skip).where(pool)) <= book_size
        size = book_size
    else:
        control_pool = pool if young_listings else pool & signal.notna()
        chosen = ranked(ranking.where(control_pool)) <= book_size
        size = book_size

    selected = chosen.shift(delay, fill_value=False)
    lagged = portfolio_construction.lag_eligibility(selected).loc[in_window]

    if rebalance_dates is None:
        starts = portfolio_construction.first_trading_days(lagged.index, frequency)
        positions = lagged.index.get_indexer(starts) + delay
        dates = lagged.index[positions[positions < len(lagged.index)]]
    else:
        dates = rebalance_dates

    return portfolio_construction.build_weights(
        lagged,
        returns.loc[mark.index <= end],
        dates,
        "equal_weight",
        1.0 / size,
        1,
    )


target_weights = build_book(TEST_START, TEST_END)
TRADE_DATES = target_weights.index
control_weights = build_book(TEST_START, TEST_END, use_momentum=False, rebalance_dates=TRADE_DATES)
null_weights = build_book(TEST_START, TEST_END, whole_pool=True, rebalance_dates=TRADE_DATES)
young_control_weights = build_book(
    TEST_START,
    TEST_END,
    use_momentum=False,
    young_listings=True,
    rebalance_dates=TRADE_DATES,
)
print(f"the rule: {len(TRADE_DATES)} rebalances, {TRADE_DATES.min().date()} to "
      f"{TRADE_DATES.max().date()}")
on_rule_dates = (
    set(control_weights.index) <= set(TRADE_DATES)
    and set(null_weights.index) <= set(TRADE_DATES)
)
print(f"the control and the null trade on the rule's dates only: {on_rule_dates}")

## 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged**,
whatever the strategy is:

- no book is more than fully invested, and none is negatively invested;
- no negative weights, if the strategy is long-only;
- **every security paid for today had its signal on at the prior close** — check the signal itself,
  not the composed eligibility, because the signal is the thing that had to exist in advance;
- every security bought is tradable on the day it is bought, so a fill price exists;
- nothing is still held on a day after it stopped being tradable.

In [ ]:
# EXAMPLE-ONLY CELL
# Cheap here, expensive inside a P&L. Every rule must pass these unchanged.
test_days = mark.index[(mark.index >= TEST_START) & (mark.index <= TEST_END)]


def book_checks(weights, label, size):
    """The invariants of one book's targets, keyed by what each one says."""
    invested = weights.sum(axis=1)
    bought = weights > 0
    member_yesterday = membership.shift(1, fill_value=False).reindex_like(bought)
    tradable_yesterday = tradable.shift(1, fill_value=False).reindex_like(bought)

    return {
        f"{label}: no book more than fully invested": bool((invested <= 1.0 + 1e-9).all()),
        f"{label}: no negative weights": bool((weights >= -1e-9).all().all()),
        f"{label}: no more names than the book holds": bool((bought.sum(axis=1) <= size).all()),
        f"{label}: every holding was in the index at the prior close": bool(
            (~bought | member_yesterday).all().all()
        ),
        f"{label}: every holding was tradable at the prior close": bool(
            (~bought | tradable_yesterday).all().all()
        ),
        f"{label}: no excluded name is ever held": bool(not bought[excluded].any().any()),
        f"{label}: trades on fewer days than it does not": bool(len(weights) < len(test_days) / 2),
    }


checks = {
    **book_checks(target_weights, "the rule", BOOK_SIZE),
    **book_checks(control_weights, "the control", BOOK_SIZE),
    **book_checks(null_weights, "the null", POOL_SIZE),
}
held_names = target_weights.gt(0)
momentum_yesterday = signal.shift(1).reindex_like(held_names)
checks["the rule: every name struck in had a momentum value at the prior close"] = bool(
    (~held_names | momentum_yesterday.notna()).all().all()
)
control_names = control_weights.gt(0)
checks["the control: every name struck in had a momentum value at the prior close"] = bool(
    (~control_names | signal.shift(1).reindex_like(control_names).notna()).all().all()
)
young_names = young_control_weights.gt(0)
YOUNG_NAME_DAYS = int((young_names & signal.shift(1).reindex_like(young_names).isna()).sum().sum())
print(f"the control with young listings struck in a name with no momentum value "
      f"{YOUNG_NAME_DAYS} times, over {len(young_control_weights)} rebalances")

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

## 3 · Construction — is this a book you would actually run?

**This is where step 4, Portfolio Construction, lives.** Four properties, each with a failure mode
a performance chart would hide:

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Measure turnover **target-to-target**. The realised figure is lower, because between rebalances the
winners drift up on their own; that calculation needs drifted weights and belongs to the backtest.

> **The blueprint's predictions about the *shape* of the book, rather than about its return, are
> settled here — before any backtest.** They are the first ones that can be wrong, and the cheapest
> to be wrong about.

## 3.1 · Write the deliverables

Two views of the same book, because two readers need it: a **long, human-readable** one with names
and classifications attached, and a **wide, identifier-keyed** `portfolio_weights.csv` — the
backtest engine's input, which looks each identifier up in the market-data folder and so has to
speak in identifiers, not in stitched positions.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; here the residual becomes a weight in the cash proxy, so the engine charges commission on
going to cash and earns the yield while there. A strategy whose defining move is *sell everything*
has to pay for it.

In [ ]:
# EXAMPLE-ONLY CELL
# Is this a book you would actually run? The target weights, held forward between rebalances --
# drift is the engine's, and this is the book's shape, not its return.
years = (TEST_END - TEST_START).days / 365.25


def describe_book(weights):
    """The book's shape over the test window, from its targets held forward."""
    daily_book = weights.reindex(test_days).ffill().fillna(0.0)
    turnover = weights.diff().abs().sum(axis=1) / 2
    held = daily_book.gt(0)
    sharadar_columns = [column for column in daily_book.columns if column in SHARADAR_IDENTIFIERS]

    return pandas.Series({
        "rebalances": len(weights),
        "rebalances per year": round(len(weights) / years, 1),
        "mean one-way turnover per rebalance": round(turnover.mean(), 3),
        "annual turnover, target to target": round(turnover.sum() / years, 2),
        "mean invested share": round(daily_book.sum(axis=1).mean(), 3),
        "lowest invested share": round(daily_book.sum(axis=1).min(), 3),
        "mean holdings": round(held.sum(axis=1).mean(), 1),
        "held name-days": int(held.sum().sum()),
        "mean weight in Sharadar names": round(daily_book[sharadar_columns].sum(axis=1).mean(), 3),
    })


construction = pandas.DataFrame({
    "the rule": describe_book(target_weights),
    "the control": describe_book(control_weights),
    "the null": describe_book(null_weights),
})
print(construction.to_string())

# Capacity: every trade's size against the name's 63-day average traded value that day.
traded_weight = target_weights.diff().abs()
traded_weight.iloc[0] = target_weights.iloc[0]
average_traded_value = traded_value.rolling(63).mean().reindex(index=target_weights.index)
room = average_traded_value.reindex(columns=target_weights.columns) / traded_weight.where(
    traded_weight > 0
)
capacity_rows = {}

for participation in (0.01, 0.05):
    trade_capacity = (room * participation).stack().dropna()
    capacity_rows[f"{participation:.0%} of a day's traded value"] = {
        "worst trade, book size in dollars": float(trade_capacity.min()),
        "1st percentile of trades": float(trade_capacity.quantile(0.01)),
        "median trade": float(trade_capacity.median()),
    }

capacity = pandas.DataFrame(capacity_rows).T
print()
print("capacity -- the largest book the rule can trade at a share of each name's traded value:")
print(capacity.map(lambda value: f"{value:,.0f}").to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# Two views of the same book. The long one is for a person; the wide one is the engine's input and
# has to speak in identifiers, because that is what its market-data folder is named by.
master = pandas.read_csv("Universe/Security_Master.csv").set_index("main_identifier")
readable = target_weights.stack()
readable = readable[readable > 0].rename("weight").reset_index()
readable.columns = ["trade_date", "position", "weight"]
readable["name"] = readable["position"].map(master["name"])
readable["provider"] = readable["position"].map(
    lambda position: "sharadar" if position in SHARADAR_IDENTIFIERS else "fmp"
)
readable.to_csv(EXPERIMENT_DIRECTORY / "Portfolio" / "holdings_readable.csv", index=False)

by_identifier = securities_panel.expand_to_identifiers(target_weights)
weight_file = backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY)
written = pandas.read_csv(weight_file, index_col=0)
print(f"{weight_file.name}: {written.shape[0]} identifiers x {written.shape[1]} rebalances")
print(f"every column sums to 1: {bool((written.sum(axis=0).sub(1.0).abs() < 1e-6).all())}")
cash_row = written.loc[backtest_engine.CASH_IDENTIFIER]
print(f"cash weight: first {cash_row.iloc[0]:.3f}, mean {cash_row.mean():.3f}")

## 4 · Backtest — KaxaNuk Backtest Engine

**In plain words:** run the rules over history, with costs, without peeking ahead.

The weight file goes to the licensed engine, which simulates the book share by share: it fills at a
real price, charges per-share commission on the unadjusted price, holds integer share counts and a
cash reserve, marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository**, and results are accepted **net** or not at all.
Clip the window to the shortest benchmark up front rather than discovering it as a crash, and report
which benchmark bound it.

> **Guard the import.** The engine installs from KaxaNuk's licensed index rather than PyPI, so this
> section reports what is missing and skips without it. Everything in `Portfolio/` is already
> written and does not depend on the engine — a clone with no licence gets a real book and no
> numbers, by design.

In [ ]:
# EXAMPLE-ONLY CELL
# Costs, the arm's, stated in BLUEPRINT_2.md rather than defaulted: the commission setting 0.1,
# which the engine charges as about eight cents a share, 5 basis points of slippage and a 2% cash
# reserve on $1,000,000. The setting 0.005 is reported beside it.
INITIAL_CAPITAL = 1_000_000
COMMISSION_CENTS = 0.1
REALISTIC_COMMISSION_CENTS = 0.005
SLIPPAGE_BASIS_POINTS = 5.0
CASH_RESERVE = 0.05
CLOSE_FILL = MARK_COLUMN
runs = {}
refused = {}


def price(
    label,
    weights,
    start,
    end,
    commission=COMMISSION_CENTS,
    execution=backtest_engine.EXECUTION_PRICE_COLUMN,
):
    """Write one book's weight file, run the engine over its window, and keep the result."""
    readable_label = label.replace(" ", "_").replace(",", "").replace("%", "")
    name = "portfolio_weights_" + readable_label.replace("-", "_")
    # A name held with no price anywhere inside the window can never be filled, and the engine
    # refuses its empty table: it is dropped from this run's weight file, its weight left in cash,
    # and named. Found when the null held WorldCom, last priced the day before the test opened.
    priced_in_window = mark.loc[start:end].notna().any()
    unpriced = [
        column
        for column in weights.columns
        if weights[column].gt(0).any() and not bool(priced_in_window.get(column, False))
    ]
    if len(unpriced) > 0:
        print(f"{label}: held with no price inside the window, dropped: {unpriced}")
    priceable = weights.drop(columns=unpriced)
    by_identifier = securities_panel.expand_to_identifiers(priceable)
    backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY, name)
    configuration = backtest_engine.build_configuration(
        EXPERIMENT_DIRECTORY,
        start.date(),
        end.date(),
        INITIAL_CAPITAL,
        commission,
        SLIPPAGE_BASIS_POINTS,
        CASH_RESERVE,
        name,
        execution_price_column=execution,
    )
    result = backtest_engine.run_backtest(EXPERIMENT_DIRECTORY, configuration)

    if not result.success:
        # Recorded by name with the engine's reason, never raised: BLUEPRINT_3.md counts a
        # sub-period or a cell the engine cannot price against the rule, and leaves claim 5
        # measured if the rule or its control cannot be priced over the test window. Found when a
        # cell held `VMW` into a month start its file has no price for, between its last trade and
        # bars that repeat it.
        refused[label] = {
            "reason": str(result.error),
            "start": start,
            "end": end,
        }
        print(f"the engine refused {label}: {result.error}")

        return

    window = backtest_engine.describe_window(result, end.date())

    if window.startswith("TRUNCATED"):
        # A run that stops valuing the book partway still returns success over the stub: the
        # blueprint excludes it by name, not priced, and counts it against the rule.
        refused[label] = {
            "reason": f"the engine stopped valuing the book, {window}",
            "start": start,
            "end": end,
        }
        print(f"the engine stopped valuing {label}: {window}")

        return

    runs[label] = {
        "result": result,
        "start": start,
        "end": end,
        "rebalances": len(weights),
        "window": window,
    }
    statistics = result.data["portfolio_stats"]
    print(f"{label}: CAGR {statistics['Annualized Return (CAGR)']:.4f}, "
          f"Sharpe {statistics['Portfolio Sharpe Ratio']:.3f}, {runs[label]['window']}")


if not backtest_engine.ENGINE_INSTALLED:
    print("step 5 skipped: the KaxaNuk Backtest Engine is not installed")
else:
    price("the rule", target_weights, TEST_START, TEST_END)
    price("the control", control_weights, TEST_START, TEST_END)
    price("the null, the whole pool", null_weights, TEST_START, TEST_END)
    price("the control with young listings", young_control_weights, TEST_START, TEST_END)
    # The owner's first window, as description: the rule and its control on 2002 to 2016 alone.
    early_book = build_book(TEST_START, EARLY_END)
    early_control = build_book(
        TEST_START,
        EARLY_END,
        use_momentum=False,
        rebalance_dates=early_book.index,
    )
    price("2002 to 2016, the rule", early_book, TEST_START, EARLY_END)
    price("2002 to 2016, the control", early_control, TEST_START, EARLY_END)

    for label, weights in (("the rule", target_weights), ("the control", control_weights)):
        price(
            f"{label}, realistic costs",
            weights,
            TEST_START,
            TEST_END,
            REALISTIC_COMMISSION_CENTS,
        )
        # The fill convention, checked: every name filled at the day's close, as Sharadar's are.
        price(
            f"{label}, every fill at the close",
            weights,
            TEST_START,
            TEST_END,
            execution=CLOSE_FILL,
        )

    # The kill switch: each sub-period priced as a window of its own, the rule entering on its first
    # rebalance, and its control held to that rule's own dates.
    for number, (sub_start, sub_end) in enumerate(SUB_PERIODS, start=1):
        sub_book = build_book(sub_start, sub_end)
        sub_control = build_book(
            sub_start,
            sub_end,
            use_momentum=False,
            rebalance_dates=sub_book.index,
        )
        price(f"sub-period {number}, the rule", sub_book, sub_start, sub_end)
        price(f"sub-period {number}, the control", sub_control, sub_start, sub_end)

    # The perturbation: one setting at a time, each cell beside its own control, which takes the
    # cell's pool, book size, frequency and delay and trades on the cell's own dates.
    for label, changes in SWEEP:
        cell_book = build_book(TEST_START, TEST_END, **changes)
        control_changes = {
            key: value
            for key, value in changes.items()
            if key in ("pool_size", "book_size", "delay")
        }
        cell_control = build_book(
            TEST_START,
            TEST_END,
            use_momentum=False,
            rebalance_dates=cell_book.index,
            **control_changes,
        )
        price(f"{label}, the rule", cell_book, TEST_START, TEST_END)
        price(f"{label}, the control", cell_control, TEST_START, TEST_END)

In [ ]:
# EXAMPLE-ONLY CELL
# Every figure below comes from the engine. There is no second simulator in this repository.
def engine_series(result):
    """The book's and the engine benchmark's daily returns, on the dates both have."""
    book_returns = result.data["Register_df"]["Returns"]
    benchmark_table = result.data["benchmark"].to_pandas()
    benchmark_returns = benchmark_table.set_index(
        pandas.to_datetime(benchmark_table["date_column"])
    )["daily_return"].astype(float)

    return pandas.concat([book_returns, benchmark_returns], axis=1, join="inner").dropna()


def deepest_fall(daily_returns, start, end):
    """The deepest fall of a value series whose peak and trough both lie inside the dates."""
    inside = daily_returns[(daily_returns.index >= start) & (daily_returns.index <= end)]
    value = (1.0 + inside).cumprod()

    return float((value / value.cummax() - 1.0).min())


if "the rule" not in runs or "the control" not in runs:
    # BLUEPRINT_3.md: the rule or its control unpriced over the test window leaves claim 5 measured.
    print(f"no summary: the rule or its control was not priced; refused: {sorted(refused)}")
else:
    rows = {}

    for label, run in runs.items():
        statistics = run["result"].data["portfolio_stats"]
        rows[label] = {
            "CAGR": statistics["Annualized Return (CAGR)"],
            "volatility": statistics["Annualized Volatility"],
            "Sharpe": statistics["Portfolio Sharpe Ratio"],
            "max drawdown": statistics["Max Drawdown"],
            "alpha vs index": statistics.get("Alpha"),
            "commissions": statistics["Total Commissions"],
            "slippage": statistics["Total Slippage Costs"],
            "rebalances": run["rebalances"],
        }

    benchmark_statistics = runs["the rule"]["result"].data["benchmark_stats"]
    rows["the index, same window"] = {
        "CAGR": benchmark_statistics["Annualized Return (CAGR)"],
        "volatility": benchmark_statistics["Annualized Volatility"],
        "Sharpe": benchmark_statistics["Portfolio Sharpe Ratio"],
        "max drawdown": benchmark_statistics["Max Drawdown"],
    }
    summary = pandas.DataFrame(rows).T
    headline_rows = [
        "the rule",
        "the control",
        "the rule, realistic costs",
        "the control, realistic costs",
        "the rule, every fill at the close",
        "the control, every fill at the close",
        "the index, same window",
        "the null, the whole pool",
        "the control with young listings",
        "2002 to 2016, the rule",
        "2002 to 2016, the control",
    ]
    pandas.set_option("display.width", 200)
    print(summary.loc[[row for row in headline_rows if row in summary.index]].to_string())
    print(f"not priced by the engine, by name: {sorted(refused) if len(refused) > 0 else 'none'}")
    print()
    # A run the engine refused counts against the rule, as the blueprint fixed: a missing figure is
    # ahead of nothing.
    UNPRICED = {
        "Sharpe": float("nan"),
        "CAGR": float("nan"),
        "rebalances": float("nan"),
    }

    def margins(rule_label, control_label):
        """The rule's margins over its control, unrounded, and if both clear; None if unpriced."""
        if rule_label not in rows or control_label not in rows:
            return None

        sharpe = rows[rule_label]["Sharpe"] - rows[control_label]["Sharpe"]
        cagr = (rows[rule_label]["CAGR"] - rows[control_label]["CAGR"]) * 100

        return {
            "Sharpe": sharpe,
            "CAGR points": cagr,
            "met": sharpe >= SHARPE_MARGIN and cagr >= CAGR_MARGIN_POINTS,
        }

    headline_margins = margins("the rule", "the control")
    close_margins = margins(
        "the rule, every fill at the close",
        "the control, every fill at the close",
    )
    realistic_margins = margins("the rule, realistic costs", "the control, realistic costs")
    early_margins = margins("2002 to 2016, the rule", "2002 to 2016, the control")
    young_margins = margins("the rule", "the control with young listings")
    MARGINS_MET = headline_margins["met"]
    SAME_VERDICT_AT_THE_CLOSE = close_margins is not None and close_margins["met"] == MARGINS_MET
    print(f"prediction 1, the rule minus its control: {headline_margins}")
    print(f"  every fill at the close: {close_margins}; "
          f"same verdict: {SAME_VERDICT_AT_THE_CLOSE}")
    print(f"  at the realistic cost setting: {realistic_margins}")
    print(f"  on 2002 to 2016 alone, description only: {early_margins}")
    print(f"  against the control with young listings, a diagnostic: {young_margins}")

    PREDICTION_2_HOLDS = rows["the rule"]["Sharpe"] > rows["the index, same window"]["Sharpe"]
    print(f"prediction 2, the rule's Sharpe above the index's: {PREDICTION_2_HOLDS}")

    rule_daily = engine_series(runs["the rule"]["result"]).iloc[:, 0]
    control_daily = engine_series(runs["the control"]["result"]).iloc[:, 0]
    in_rebound = (rule_daily.index >= REBOUND_START) & (rule_daily.index <= REBOUND_END)
    rule_rebound = float((1.0 + rule_daily[in_rebound]).prod() - 1.0)
    control_window = (control_daily.index >= REBOUND_START) & (control_daily.index <= REBOUND_END)
    control_rebound = float((1.0 + control_daily[control_window]).prod() - 1.0)
    PREDICTION_3_HOLDS = rule_rebound < control_rebound
    print(f"prediction 3, from {REBOUND_START.date()} to {REBOUND_END.date()}: the rule "
          f"{rule_rebound:.4f}, the control {control_rebound:.4f}; "
          f"the rule trails: {PREDICTION_3_HOLDS}")

    for label in ("the rule", "the control"):
        paired = engine_series(runs[label]["result"])
        beta = paired.iloc[:, 0].cov(paired.iloc[:, 1]) / paired.iloc[:, 1].var()
        print(f"beta to the index, {label}: {beta:.3f}")

    # The kill switch: fewer than two sub-periods with the rule ahead on both is a failure.
    sub_rows = []

    for number in range(1, len(SUB_PERIODS) + 1):
        rule_row = rows.get(f"sub-period {number}, the rule", UNPRICED)
        control_row = rows.get(f"sub-period {number}, the control", UNPRICED)
        sub_rows.append({
            "sub-period": number,
            "Sharpe, the rule": rule_row["Sharpe"],
            "Sharpe, the control": control_row["Sharpe"],
            "CAGR, the rule": rule_row["CAGR"],
            "CAGR, the control": control_row["CAGR"],
            "the rule ahead on both": (
                rule_row["Sharpe"] > control_row["Sharpe"]
                and rule_row["CAGR"] > control_row["CAGR"]
            ),
        })

    kill_switch = pandas.DataFrame(sub_rows).set_index("sub-period")
    print()
    print(kill_switch.to_string())
    SUB_PERIODS_AHEAD = int(kill_switch["the rule ahead on both"].sum())
    KILL_SWITCH_TRIPS = (not MARGINS_MET) or SUB_PERIODS_AHEAD < 2
    print(f"sub-periods with the rule ahead of its control on both: {SUB_PERIODS_AHEAD} of 3")
    print(f"the kill switch trips: {KILL_SWITCH_TRIPS}")

    # The perturbation: a cell counts for the rule only when it is ahead on Sharpe and on CAGR.
    sweep_rows = []

    for label, _ in SWEEP:
        rule_row = rows.get(f"{label}, the rule", UNPRICED)
        control_row = rows.get(f"{label}, the control", UNPRICED)
        sweep_rows.append({
            "cell": label,
            "Sharpe": rule_row["Sharpe"],
            "Sharpe over control": rule_row["Sharpe"] - control_row["Sharpe"],
            "CAGR over control, points": (rule_row["CAGR"] - control_row["CAGR"]) * 100,
            "rebalances": rule_row["rebalances"],
        })

    sweep_table = pandas.DataFrame(sweep_rows).set_index("cell")
    sweep_table["ahead on both"] = (
        (sweep_table["Sharpe over control"] > 0) & (sweep_table["CAGR over control, points"] > 0)
    )
    CELLS_AHEAD = int(sweep_table["ahead on both"].sum())
    CRITERION_3_READ_AS_PASSED = CELLS_AHEAD >= CELLS_REQUIRED
    print()
    print(sweep_table.to_string())
    print(f"cells with the rule ahead of its control on both: {CELLS_AHEAD} of {len(SWEEP)}; "
          f"criterion 3 read as passed: {CRITERION_3_READ_AS_PASSED}")

In [ ]:
# EXAMPLE-ONLY CELL
# Watched, not predicted: the Sharadar names' share of each book, every name held on its last priced
# day, and each window read on its own years, from the engine's own daily weights and returns.
if len(runs) > 0:
    last_priced = mark.apply(lambda column: column.last_valid_index())

    for label in ("the rule", "the control"):
        daily = backtest_engine.read_daily_weights(runs[label]["result"])
        daily.index = pandas.to_datetime(daily.index)
        sharadar_columns = [column for column in daily.columns if column in SHARADAR_IDENTIFIERS]
        sharadar_share = daily[sharadar_columns].sum(axis=1)
        print(f"{label}: weight in Sharadar names, mean {sharadar_share.mean():.3f}, "
              f"highest {sharadar_share.max():.3f}")
        held_to_the_end = []

        for column in daily.columns:
            if column not in last_priced.index or pandas.isna(last_priced[column]):
                continue

            final_day = last_priced[column]

            if final_day > TEST_END or final_day not in daily.index:
                continue

            final_weight = float(daily.at[final_day, column])

            if final_weight > 0:
                held_to_the_end.append((column, final_day.date(), round(final_weight, 4)))

        print(f"{label}: held on its last priced day ({len(held_to_the_end)}): {held_to_the_end}")

## 5 · Attribution — KaxaNuk Attribution Analysis

**In plain words:** which part of the return did you actually earn?

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler**, the first cut: active return into an **allocation** effect — being
  overweight the right groups — and a **selection** effect, picking the right securities inside
  them. The exact lever that moved.
- **A factor model**, the second layer: excess return into **compensated factor tilts** — beta,
  momentum, residual volatility, liquidity — and **idiosyncratic** alpha, what was earned on
  purpose rather than by accident.
- **Brinson-Fachler again, on the residual**, the third pass: the selection story sharpens, and
  it says whether the Sharpe survives once the factor turns.

**What it settles and what it does not** is in [`../../AGENTS.md`](../../AGENTS.md) — including the
four counterfactual books that answer what the factor model cannot.

**Two inputs are supplied by hand**, from `Data/Curator/Benchmarks/` and `Data/Curator/Factors/` —
the benchmark's weights and returns, and the factor returns. No price provider sells them. Getting
their layout wrong makes the loader read the attribution transposed rather than fail, so shape them
in one place and say what is missing before trying.

**The book arrives daily.** The attribution library rejects a weight file that is not a daily
series, so it reads the book as the engine held it each trading day, drift included, from
`Backtest/` — never `portfolio_weights.csv`, which holds only the rebalance dates.

**What binds the window.** The attribution period is the intersection of the factor files and the
benchmark holdings, so it is usually *shorter* than the backtest. The two sets of numbers describe
different periods and must not be compared directly. Record both windows in `FINDINGS_3.md`.

In [ ]:
# EXAMPLE-ONLY CELL
missing = attribution_analysis.report_missing_inputs()

if len(runs) == 0:
    missing.append("no backtest to attribute: step 5 was skipped")

ATTRIBUTION_READY = len(missing) == 0

if not ATTRIBUTION_READY:
    print("step 6 skipped:", "; ".join(missing))
else:
    import kaxanuk.attribution_analysis.performance_attribution

    benchmark_weights_by_run = {}
    factor_returns = attribution_analysis.load_factor_returns()
    factor_dates = factor_returns["f_market"].index


    def widened(label):
        """One run's daily book, widened to the whole index, and the index beside it."""
        daily_weights = backtest_engine.read_daily_weights(runs[label]["result"])
        benchmark_weights = attribution_analysis.load_benchmark_weights(daily_weights.index)
        book = attribution_analysis.widen_to_benchmark(
            daily_weights,
            benchmark_weights,
            backtest_engine.BENCHMARK_IDENTIFIER,
        )
        benchmark = benchmark_weights.reindex(columns=book.columns).fillna(0.0)
        book = book.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
        benchmark = benchmark.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
        benchmark = benchmark.div(benchmark.sum(axis=1), axis=0).fillna(0.0)

        return book, benchmark


    book, benchmark = widened("the rule")
    universe = tuple(book.columns)
    asset_returns = attribution_analysis.load_asset_returns(universe, book.index)
    window = book.index[(book.index >= factor_dates.min()) & (book.index <= factor_dates.max())]
    print(f"backtest window:    {book.index.min().date()} to {book.index.max().date()}")
    print(f"attribution window: {window.min().date()} to {window.max().date()}, bound by the "
          f"factor files' first date")
    print(f"securities priced: {asset_returns.shape[1]} of {len(universe)} the two books name")

In [ ]:
# EXAMPLE-ONLY CELL
# First cut: Brinson-Fachler. Active return into allocation, selection and interaction.
if ATTRIBUTION_READY:
    brinson = kaxanuk.attribution_analysis.performance_attribution.BrinstonFachlerArrowAttribution(
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        attribution_analysis.to_arrow(book.loc[window]),
        attribution_analysis.to_arrow(benchmark.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    brinson.time_series_calculation()
    brinson_daily = brinson.df.to_pandas().set_index("date")
    brinson_totals = brinson_daily[["alpha", "allocation", "selection", "interaction"]].sum() * 100
    print("Brinson-Fachler, the rule, summed over the window, in percentage points:")
    print(brinson_totals.round(2).to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# Second layer: the factor model, on the rule and on its control, so the cross's own share of the
# idiosyncratic return is the difference between the two books that differ in it alone.
if ATTRIBUTION_READY:
    by_factor = {
        name: attribution_analysis.to_arrow(frame.loc[window.min():window.max()])
        for name, frame in factor_returns.items()
    }
    decomposition_columns = {}

    for label in ("the rule", "the control"):
        arm_book, _ = widened(label)
        arm_book = arm_book.reindex(columns=list(universe), fill_value=0.0)
        factor_model = kaxanuk.attribution_analysis.performance_attribution.KNFMArrowAttribution(
            attribution_analysis.to_arrow(arm_book.loc[window]),
            by_factor,
            attribution_analysis.to_arrow(asset_returns.loc[window]),
            date_column=attribution_analysis.DATE_HEADER,
        )
        factor_model.multifactor_attribution()
        factor_daily = factor_model.portfolio_attribution_ts.to_pandas()
        decomposition_columns[label] = factor_daily.select_dtypes("number").sum() * 100

    decomposition = pandas.DataFrame(decomposition_columns)
    print("Factor model, summed over the attribution window, in percentage points:")
    print(decomposition.round(2).to_string())

## 6 · Counterfactuals — who earned the idiosyncratic share

The factor model leaves part of the book's excess return unexplained. That is a number, not an
answer: the book makes choices the index does not. **Each counterfactual removes exactly one of
those and keeps the rest**, which is the only way the question stops being an inference. The
engine can already price all of them; `AGENTS.md`, under *What attribution must report*, names four
follow-ups.

In [ ]:
# EXAMPLE-ONLY CELL
# The control differs from the rule in the momentum ranking alone, so the gap in their
# idiosyncratic points is the ranking's share; the null, the whole pool, is described beside them.
if ATTRIBUTION_READY:
    idiosyncratic = decomposition.loc["f_idyo_returns"]
    print(f"idiosyncratic points: the rule {idiosyncratic['the rule']:.2f}, "
          f"the control {idiosyncratic['the control']:.2f}, the ranking's share "
          f"{idiosyncratic['the rule'] - idiosyncratic['the control']:+.2f}")

## 7 · Verdict

**In words.** A notebook that ends in a number and no sentence gets read as whatever the reader
hoped.

Three sentences: does the book work; what attribution says about why; what the next experiment
should change. Then copy the numbers into [`FINDINGS_3.md`](FINDINGS_3.md) — **that file is the
record, this notebook is the method.**

If sections 4 and 5 reported "not installed", this notebook has produced a book and no result,
which is the honest outcome and not a failure.

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine |
| `Portfolio/` — the readable book and the summaries | humans, and `FINDINGS_3.md` |
| `Backtest/` — the track record, and the book's daily weights | the attribution library, `FINDINGS_3.md`, and the comparison baseline for every later experiment |
| `Attribution/` | `FINDINGS_3.md`, and the comparison baseline for every later experiment |

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** | Without them the book is built but its performance is not measured, and the blueprint's return predictions stay open. |
| 2 | **Turnover is target-to-target, not realised.** | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by the supplied files' coverage. | The two sets of numbers describe different periods. |
| 4 | **Delisting exits use one day of hindsight.** | Inert on a universe of live securities; load-bearing on any universe that retains delisted names. |
| 5 | **Cash is a real instrument**, so going flat costs commission and earns a yield. | A strategy that trades to cash often is partly a bet on the front end of the curve. Ask attribution about it. |
| 6 | **Every lever the benchmark declines** — a weight cap, a minimum holding count, risk-aware sizing — is a later experiment, and each has to beat this book to earn its place. | Complexity is added one lever at a time. |

In [ ]:
# EXAMPLE-ONLY CELL
# The verdict is written in FINDINGS_3.md from what this notebook printed; this cell prints the
# comparisons the blueprint fixed, from the engine's own figures, and nothing else.
if "the rule" in runs and "the control" in runs:
    print(f"the test window: {TEST_START.date()} to {TEST_END.date()}")
    print(f"prediction 1, margins met: {MARGINS_MET}; the same verdict with every fill at the "
          f"close: {SAME_VERDICT_AT_THE_CLOSE}")
    print(f"prediction 2, the rule's Sharpe above the index's: {PREDICTION_2_HOLDS}")
    print(f"prediction 3, the rule trails its control after the 2009 low: {PREDICTION_3_HOLDS}")
    print(f"the kill switch: ahead on both in {SUB_PERIODS_AHEAD} of 3 sub-periods; trips: "
          f"{KILL_SWITCH_TRIPS}")
    print(f"criterion 3: {CELLS_AHEAD} of {len(SWEEP)} cells ahead on both; passed: "
          f"{CRITERION_3_READ_AS_PASSED}")
    print(f"runs the engine did not price, each counted against the rule: {sorted(refused)}")
    CLAIM_5_CONFIRMED = (
        MARGINS_MET
        and not KILL_SWITCH_TRIPS
        and SAME_VERDICT_AT_THE_CLOSE
        and PREDICTION_2_HOLDS
    )
    print(f"claim 5 confirmed as a book on the test window: {CLAIM_5_CONFIRMED}")

## 8 · Verify

**Assertions that raise when this experiment's output is wrong.** Section 2.1 prints its
invariants; this section raises on them, and on what the notebook wrote, so a run that reaches the
last cell is one whose book and numbers hold. At the least:

- every invariant of section 2.1 holds;
- `Portfolio/portfolio_weights.csv`, read back, has one column per rebalance date, each summing to
  one with the cash proxy, and no negative weight;
- **every engine run valued the window it was asked for**: the days it valued, counted against the
  trading days in that window — never against a fixed floor, which a run that stopped years early
  can clear. Nearly every trading day valued, and none missing from the window's end, where a
  truncated run loses them. Skipped, as section 4 is, when the engine is not installed;
- the attribution window lies inside the backtest's, and every arm has an idiosyncratic figure.
  Skipped, as section 5 is, when attribution did not run.

In [ ]:
# EXAMPLE-ONLY CELL
# The books: the invariants section 2.1 printed, raised here, and the weight file read back from
# disk rather than from the frame that wrote it.
written_weights = pandas.read_csv(weight_file, index_col=0)
column_totals = written_weights.sum(axis=0)
book_verifications = {
    **checks,
    "one column per rebalance": written_weights.shape[1] == len(TRADE_DATES),
    "every column sums to one, cash included": bool(
        ((column_totals - 1.0).abs() <= 0.0001).all()
    ),
    "the cash proxy has a row": backtest_engine.CASH_IDENTIFIER in written_weights.index,
    "no negative weight": bool((written_weights >= 0.0).all(axis=None)),
    "no weight file reaches past the test window": bool(
        pandas.to_datetime(written_weights.columns).max() <= TEST_END
    ),
    "the null trades on the rule's dates only": set(null_weights.index) <= set(TRADE_DATES),
    "the control trades on the rule's dates only": set(control_weights.index) <= set(TRADE_DATES),
}
failed_book = [
    name
    for name, passed in book_verifications.items()
    if not passed
]

if len(failed_book) > 0:
    message = f"the books failed verification: {'; '.join(failed_book)}"

    raise AssertionError(message)

print(f"verified: {len(book_verifications)} checks on the books and {weight_file.name}")

In [ ]:
# EXAMPLE-ONLY CELL
# Every engine run, counted against the trading days of the window it was asked for. A run that
# stops valuing the book still reports success and summarises the stub; the window's own days are
# the check. A day or two at the edges is the engine's calendar, not a truncation.
VALUED_SHARE_REQUIRED = 0.99
UNVALUED_DAYS_AT_END_ALLOWED = 5
run_verifications = {}

for label, run in runs.items():
    run_days = mark.index[(mark.index >= run["start"]) & (mark.index <= run["end"])]
    valued_days = pandas.DatetimeIndex(backtest_engine.read_daily_weights(run["result"]).index)
    valued_in_window = run_days.intersection(valued_days)
    valued_share = len(valued_in_window) / len(run_days)
    unvalued_at_end = run_days[run_days > valued_days.max()]
    run_verifications[f"{label}: {valued_share:.1%} of the window's trading days valued"] = (
        valued_share >= VALUED_SHARE_REQUIRED
    )
    run_verifications[f"{label}: {len(unvalued_at_end)} trading days unvalued at the end"] = (
        len(unvalued_at_end) <= UNVALUED_DAYS_AT_END_ALLOWED
    )

for label, refusal in refused.items():
    run_verifications[f"{label}: not priced, named with its reason"] = bool(
        refusal["reason"]
    )

if len(runs) == 0:
    print("engine checks skipped, as section 4 was: the KaxaNuk Backtest Engine is not installed")

if ATTRIBUTION_READY:
    run_verifications["the attribution window lies inside the backtest's"] = bool(
        len(window) > 0
        and window.min() >= book.index.min()
        and window.max() <= book.index.max()
    )
    run_verifications["both books have an idiosyncratic figure"] = bool(
        decomposition.loc["f_idyo_returns"].notna().all()
    )

failed_runs = [
    name
    for name, passed in run_verifications.items()
    if not passed
]

if len(failed_runs) > 0:
    message = f"the runs failed verification: {'; '.join(failed_runs)}"

    raise AssertionError(message)

print(f"verified: {len(run_verifications)} checks on {len(runs)} engine runs; not priced, by name: "
      f"{len(refused)}")